# 03 — Predictive Modeling

This notebook evaluates whether engineered limit-order-book and order-flow features contain out-of-sample information about future mid-price returns.

Models are developed using the March 16–18 training period and evaluated for model selection on March 19. March 20 is reserved as the final test period.

The modeling strategy proceeds from simple benchmarks to increasingly flexible models:

1. Naive return benchmarks
2. Linear regression
3. Regularized linear models
4. Nonlinear / machine-learning models

This progression helps determine whether additional model complexity produces genuine predictive improvement.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROCESSED_DIR = Path("../data/processed")

## 1. Load Processed Data

Load the independently constructed asset-day datasets and recreate the chronological train, validation, and test partitions.

In [2]:
assets = ["TSLA", "AMZN", "BRK.B"]

train_dates = [
    "2026-03-16",
    "2026-03-17",
    "2026-03-18",
]

val_dates = ["2026-03-19"]
test_dates = ["2026-03-20"]

model_features = [
    "spread",
    "imbalance_l1",
    "imbalance_l5",
    "bid_depth_5",
    "ask_depth_5",
    "microprice_deviation",

    "event_count",
    "execution_count",
    "normalized_book_flow",
    "execution_imbalance",

    "return_500ms",
    "return_1s",
    "return_2s",
    "return_5s",
    "return_10s",

    "volatility_5s",
    "volatility_30s",
    "avg_events_5s",
    "avg_executions_5s",
]

target_cols = {
    "TSLA": [
        "target_500ms",
        "target_2s",
        "target_10s",
    ],
    "AMZN": [
        "target_500ms",
        "target_2s",
        "target_10s",
    ],
    "BRK.B": [
        "target_5s",
        "target_10s",
        "target_30s",
    ],
}

In [3]:
def load_processed_day(asset, date):
    safe_asset = asset.replace(".", "_")

    path = (
        PROCESSED_DIR /
        f"{safe_asset}_{date}.parquet"
    )

    return pd.read_parquet(path)


def combine_processed_dates(asset, dates):
    return pd.concat(
        [
            load_processed_day(asset, date)
            for date in dates
        ],
        axis=0
    )

In [4]:
splits = {}

for asset in assets:
    splits[asset] = {
        "train": combine_processed_dates(
            asset, train_dates
        ),
        "val": combine_processed_dates(
            asset, val_dates
        ),
        "test": combine_processed_dates(
            asset, test_dates
        ),
    }

    print(
        asset,
        "Train:", splits[asset]["train"].shape,
        "Validation:", splits[asset]["val"].shape,
        "Test:", splits[asset]["test"].shape,
    )

TSLA Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)
AMZN Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)
BRK.B Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)


## 2. Regression Baselines

Before fitting learned models, we establish a naive persistence benchmark.

Because the regression target is the future log mid-price return,

$$
y_{t,h}=\log\left(\frac{m_{t+h}}{m_t}\right),
$$

a persistence forecast assumes that the future mid-price remains equal to the current mid-price:

$$
\hat m_{t+h}=m_t.
$$

Therefore, the corresponding predicted return is

$$
\hat y_{t,h}=0.
$$

All learned regression models will be compared against this benchmark.

In [5]:
def make_xy(df, feature_cols, target_col):
    valid = (
        df[feature_cols].notna().all(axis=1)
        & df[target_col].notna()
    )

    X = df.loc[valid, feature_cols].copy()
    y = df.loc[valid, target_col].copy()

    return X, y

In [6]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    directional_accuracy = np.mean(
        np.sign(y_true)
        == np.sign(y_pred)
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Directional_Accuracy": directional_accuracy,
    }

In [7]:
baseline_results = []

for asset in assets:
    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            model_features,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            model_features,
            target
        )

        # Persistence in price space corresponds
        # to zero future return.
        y_pred = np.zeros(len(y_val))

        metrics = regression_metrics(
            y_val,
            y_pred
        )

        baseline_results.append({
            "asset": asset,
            "target": target,
            "model": "Persistence",
            **metrics,
        })


baseline_results_df = pd.DataFrame(
    baseline_results
)

display(baseline_results_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Persistence,0.000055,0.000086,-0.000003,0.156080
1,TSLA,target_2s,Persistence,0.000120,0.000175,-0.000012,0.050026
2,TSLA,target_10s,Persistence,0.000273,0.000381,-0.000056,0.018236
3,AMZN,target_500ms,Persistence,0.000045,0.000073,-0.000012,0.307773
4,AMZN,target_2s,Persistence,0.000098,0.000143,-0.000050,0.115436
5,AMZN,target_10s,Persistence,0.000225,0.000317,-0.000228,0.041674
6,BRK.B,target_5s,Persistence,0.000078,0.000139,-0.000041,0.345795
7,BRK.B,target_10s,Persistence,0.000123,0.000192,-0.000083,0.170313
8,BRK.B,target_30s,Persistence,0.000230,0.000321,-0.000247,0.026028


In [8]:
mean_baseline_results = []

for asset in assets:
    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            model_features,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            model_features,
            target
        )

        train_mean = y_train.mean()

        y_pred = np.full(
            len(y_val),
            train_mean
        )

        metrics = regression_metrics(
            y_val,
            y_pred
        )

        mean_baseline_results.append({
            "asset": asset,
            "target": target,
            "model": "Train Mean",
            **metrics,
        })


mean_baseline_results_df = pd.DataFrame(
    mean_baseline_results
)

display(mean_baseline_results_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Train Mean,0.000055,0.000086,-0.000001,0.427651
1,TSLA,target_2s,Train Mean,0.000120,0.000175,-0.000005,0.476292
2,TSLA,target_10s,Train Mean,0.000273,0.000381,-0.000027,0.492080
3,AMZN,target_500ms,Train Mean,0.000045,0.000073,-0.000011,0.343161
4,AMZN,target_2s,Train Mean,0.000098,0.000143,-0.000047,0.437586
5,AMZN,target_10s,Train Mean,0.000225,0.000317,-0.000196,0.484225
6,BRK.B,target_5s,Train Mean,0.000078,0.000139,-0.000006,0.344083
7,BRK.B,target_10s,Train Mean,0.000123,0.000192,-0.000014,0.431079
8,BRK.B,target_30s,Train Mean,0.000230,0.000321,-0.000097,0.500621


In [9]:
baseline_comparison = pd.concat(
    [
        baseline_results_df,
        mean_baseline_results_df
    ],
    ignore_index=True
)

display(
    baseline_comparison.sort_values(
        ["asset", "target", "model"]
    )
)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
5,AMZN,target_10s,Persistence,0.000225,0.000317,-0.000228,0.041674
14,AMZN,target_10s,Train Mean,0.000225,0.000317,-0.000196,0.484225
4,AMZN,target_2s,Persistence,0.000098,0.000143,-0.000050,0.115436
13,AMZN,target_2s,Train Mean,0.000098,0.000143,-0.000047,0.437586
3,AMZN,target_500ms,Persistence,0.000045,0.000073,-0.000012,0.307773
12,AMZN,target_500ms,Train Mean,0.000045,0.000073,-0.000011,0.343161
7,BRK.B,target_10s,Persistence,0.000123,0.000192,-0.000083,0.170313
16,BRK.B,target_10s,Train Mean,0.000123,0.000192,-0.000014,0.431079
8,BRK.B,target_30s,Persistence,0.000230,0.000321,-0.000247,0.026028
17,BRK.B,target_30s,Train Mean,0.000230,0.000321,-0.000097,0.500621


## 3. Linear Models

Linear regression provides an interpretable first test of whether the engineered
microstructure features contain predictive information beyond the naive benchmarks.

Because several predictors are correlated and have substantially different numerical
scales, Ridge regression is also evaluated. Feature standardization is fitted using
training data only and then applied unchanged to the validation set.

In [10]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler

In [11]:
linear_results = []

for asset in assets:
    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            model_features,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            model_features,
            target
        )

        # Fit scaler using TRAINING DATA ONLY
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # Linear regression
        model = LinearRegression()
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_val_scaled)

        metrics = regression_metrics(y_val, y_pred)

        linear_results.append({
            "asset": asset,
            "target": target,
            "model": "Linear Regression",
            **metrics
        })


linear_results_df = pd.DataFrame(linear_results)

display(linear_results_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Linear Regression,0.000055,0.000086,0.002389,0.453818
1,TSLA,target_2s,Linear Regression,0.000120,0.000175,-0.000661,0.490928
2,TSLA,target_10s,Linear Regression,0.000273,0.000383,-0.008457,0.500664
3,AMZN,target_500ms,Linear Regression,0.000046,0.000073,0.009656,0.385395
4,AMZN,target_2s,Linear Regression,0.000098,0.000143,0.002910,0.467113
5,AMZN,target_10s,Linear Regression,0.000225,0.000316,0.002327,0.497346
6,BRK.B,target_5s,Linear Regression,0.000080,0.000139,0.007846,0.351059
7,BRK.B,target_10s,Linear Regression,0.000124,0.000192,0.004449,0.434867
8,BRK.B,target_30s,Linear Regression,0.000231,0.000321,-0.000538,0.497793


In [12]:
ridge_results = []

ridge_alphas = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0
]

for asset in assets:
    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            model_features,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            model_features,
            target
        )

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        for alpha in ridge_alphas:

            model = Ridge(alpha=alpha)

            model.fit(
                X_train_scaled,
                y_train
            )

            y_pred = model.predict(
                X_val_scaled
            )

            metrics = regression_metrics(
                y_val,
                y_pred
            )

            ridge_results.append({
                "asset": asset,
                "target": target,
                "model": "Ridge",
                "alpha": alpha,
                **metrics
            })


ridge_results_df = pd.DataFrame(
    ridge_results
)

display(ridge_results_df)

,asset,target,model,alpha,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Ridge,0.01,0.000055,0.000086,0.002389,0.453818
1,TSLA,target_500ms,Ridge,0.10,0.000055,0.000086,0.002389,0.453818
2,TSLA,target_500ms,Ridge,1.00,0.000055,0.000086,0.002389,0.453839
3,TSLA,target_500ms,Ridge,10.00,0.000055,0.000086,0.002389,0.453839
4,TSLA,target_500ms,Ridge,100.00,0.000055,0.000086,0.002388,0.453861
5,TSLA,target_2s,Ridge,0.01,0.000120,0.000175,-0.000661,0.490928
6,TSLA,target_2s,Ridge,0.10,0.000120,0.000175,-0.000661,0.490928
7,TSLA,target_2s,Ridge,1.00,0.000120,0.000175,-0.000661,0.490928
8,TSLA,target_2s,Ridge,10.00,0.000120,0.000175,-0.000661,0.490885
9,TSLA,target_2s,Ridge,100.00,0.000120,0.000175,-0.000658,0.491056


In [13]:
best_ridge = (
    ridge_results_df
    .sort_values("RMSE")
    .groupby(
        ["asset", "target"],
        as_index=False
    )
    .first()
)

display(best_ridge)

,asset,target,model,alpha,MAE,RMSE,R2,Directional_Accuracy
0,AMZN,target_10s,Ridge,100.00,0.000225,0.000316,0.002330,0.497282
1,AMZN,target_2s,Ridge,100.00,0.000098,0.000143,0.002911,0.467135
2,AMZN,target_500ms,Ridge,0.01,0.000046,0.000073,0.009656,0.385395
3,BRK.B,target_10s,Ridge,100.00,0.000124,0.000192,0.004467,0.434889
4,BRK.B,target_30s,Ridge,100.00,0.000231,0.000321,-0.000528,0.497622
5,BRK.B,target_5s,Ridge,100.00,0.000080,0.000139,0.007855,0.351145
6,TSLA,target_10s,Ridge,100.00,0.000273,0.000383,-0.008428,0.500621
7,TSLA,target_2s,Ridge,100.00,0.000120,0.000175,-0.000658,0.491056
8,TSLA,target_500ms,Ridge,0.01,0.000055,0.000086,0.002389,0.453818


In [14]:
comparison = pd.concat(
    [
        baseline_comparison,
        linear_results_df,
        best_ridge
    ],
    ignore_index=True
)

comparison = comparison.sort_values(
    ["asset", "target", "RMSE"]
)

display(
    comparison[
        [
            "asset",
            "target",
            "model",
            "alpha",
            "MAE",
            "RMSE",
            "R2",
            "Directional_Accuracy"
        ]
    ]
)

,asset,target,model,alpha,MAE,RMSE,R2,Directional_Accuracy
27,AMZN,target_10s,Ridge,100.00,0.000225,0.000316,0.002330,0.497282
23,AMZN,target_10s,Linear Regression,NaN,0.000225,0.000316,0.002327,0.497346
14,AMZN,target_10s,Train Mean,NaN,0.000225,0.000317,-0.000196,0.484225
5,AMZN,target_10s,Persistence,NaN,0.000225,0.000317,-0.000228,0.041674
28,AMZN,target_2s,Ridge,100.00,0.000098,0.000143,0.002911,0.467135
22,AMZN,target_2s,Linear Regression,NaN,0.000098,0.000143,0.002910,0.467113
13,AMZN,target_2s,Train Mean,NaN,0.000098,0.000143,-0.000047,0.437586
4,AMZN,target_2s,Persistence,NaN,0.000098,0.000143,-0.000050,0.115436
21,AMZN,target_500ms,Linear Regression,NaN,0.000046,0.000073,0.009656,0.385395
29,AMZN,target_500ms,Ridge,0.01,0.000046,0.000073,0.009656,0.385395


### Linear-model observations

The validation results indicate that predictive performance is strongly asset- and
horizon-dependent.

- AMZN exhibits the clearest linear predictability, particularly at the 500 ms horizon.
- BRK.B shows modest predictive information at 5 s and 10 s, but not at 30 s.
- TSLA shows a small improvement at 500 ms, while the 2 s and 10 s models fail to
  outperform the naive return benchmark in squared-error terms.
- Ridge and ordinary least squares produce nearly identical results, suggesting that
  multicollinearity is not the primary limitation of the linear models.

Overall, the small out-of-sample R² values suggest that the engineered features contain
some short-horizon information, but much of the relationship may be nonlinear or
regime-dependent.

## 4. Nonlinear Gradient-Boosting Model

To test whether the relationship between order-book state, order flow, and future
returns is nonlinear, a histogram-based gradient-boosting regressor is evaluated.

Unlike the linear models, tree-based boosting can capture nonlinear effects and
interactions between features such as spread, imbalance, depth, volatility, and
order-flow pressure. Feature scaling is not required.

In [15]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [16]:
hgb_results = []

for asset in assets:
    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            model_features,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            model_features,
            target
        )

        model = HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        metrics = regression_metrics(
            y_val,
            y_pred
        )

        hgb_results.append({
            "asset": asset,
            "target": target,
            "model": "HistGradientBoosting",
            **metrics
        })


hgb_results_df = pd.DataFrame(hgb_results)

display(hgb_results_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,HistGradientBoosting,0.000055,0.000086,0.001273,0.452406
1,TSLA,target_2s,HistGradientBoosting,0.000120,0.000175,-0.000104,0.487376
2,TSLA,target_10s,HistGradientBoosting,0.000273,0.000381,-0.000972,0.495184
3,AMZN,target_500ms,HistGradientBoosting,0.000046,0.000073,0.004209,0.382272
4,AMZN,target_2s,HistGradientBoosting,0.000098,0.000143,0.000569,0.468183
5,AMZN,target_10s,HistGradientBoosting,0.000225,0.000316,0.003179,0.496554
6,BRK.B,target_5s,HistGradientBoosting,0.000078,0.000140,-0.004325,0.348235
7,BRK.B,target_10s,HistGradientBoosting,0.000123,0.000192,0.003591,0.437286
8,BRK.B,target_30s,HistGradientBoosting,0.000231,0.000322,-0.008799,0.498136


In [17]:
ridge_vs_hgb = pd.concat(
    [
        best_ridge,
        hgb_results_df
    ],
    ignore_index=True
)

display(
    ridge_vs_hgb[
        [
            "asset",
            "target",
            "model",
            "MAE",
            "RMSE",
            "R2",
            "Directional_Accuracy"
        ]
    ].sort_values(
        ["asset", "target", "RMSE"]
    )
)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
14,AMZN,target_10s,HistGradientBoosting,0.000225,0.000316,0.003179,0.496554
0,AMZN,target_10s,Ridge,0.000225,0.000316,0.002330,0.497282
1,AMZN,target_2s,Ridge,0.000098,0.000143,0.002911,0.467135
13,AMZN,target_2s,HistGradientBoosting,0.000098,0.000143,0.000569,0.468183
2,AMZN,target_500ms,Ridge,0.000046,0.000073,0.009656,0.385395
12,AMZN,target_500ms,HistGradientBoosting,0.000046,0.000073,0.004209,0.382272
3,BRK.B,target_10s,Ridge,0.000124,0.000192,0.004467,0.434889
16,BRK.B,target_10s,HistGradientBoosting,0.000123,0.000192,0.003591,0.437286
4,BRK.B,target_30s,Ridge,0.000231,0.000321,-0.000528,0.497622
17,BRK.B,target_30s,HistGradientBoosting,0.000231,0.000322,-0.008799,0.498136


### Nonlinear-model observations

The histogram gradient-boosting model does not consistently outperform the linear
Ridge model on the validation day.

- Ridge performs better for the strongest short-horizon cases, including AMZN at
  500 ms and BRK.B at 5 s.
- Gradient boosting improves some weaker cases, such as TSLA at longer horizons,
  but the resulting predictive performance remains limited.
- The absence of a consistent nonlinear-model advantage suggests that simply
  increasing model complexity is not sufficient to substantially improve prediction.

Therefore, extensive hyperparameter tuning of the nonlinear model is not pursued.
The analysis instead focuses on understanding which features contain predictive
information and evaluating the selected models on the held-out test day.

## 5. Feature Analysis

The fitted linear models are examined to determine which order-book and order-flow
features contribute most strongly to short-horizon return prediction.

Because the features have different numerical scales, standardized Ridge
coefficients are used for comparison.

In [18]:
ridge_coef_results = []

for asset in assets:
    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            model_features,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            model_features,
            target
        )

        # Use the alpha selected previously on validation
        alpha = best_ridge.loc[
            (best_ridge["asset"] == asset) &
            (best_ridge["target"] == target),
            "alpha"
        ].iloc[0]

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)

        model = Ridge(alpha=alpha)
        model.fit(X_train_scaled, y_train)

        for feature, coef in zip(model_features, model.coef_):
            ridge_coef_results.append({
                "asset": asset,
                "target": target,
                "feature": feature,
                "coefficient": coef,
                "abs_coefficient": abs(coef)
            })


ridge_coef_df = pd.DataFrame(ridge_coef_results)

In [19]:
top_features = (
    ridge_coef_df
    .sort_values(
        ["asset", "target", "abs_coefficient"],
        ascending=[True, True, False]
    )
    .groupby(
        ["asset", "target"],
        as_index=False
    )
    .head(8)
)

display(top_features)

,asset,target,feature,coefficient,abs_coefficient
112,AMZN,target_10s,avg_events_5s,1.302753e-05,1.302753e-05
108,AMZN,target_10s,return_5s,-9.816777e-06,9.816777e-06
110,AMZN,target_10s,volatility_5s,-8.432493e-06,8.432493e-06
97,AMZN,target_10s,imbalance_l5,8.246185e-06,8.246185e-06
95,AMZN,target_10s,spread,4.825701e-06,4.825701e-06
...,...,...,...,...,...
14,TSLA,target_500ms,return_10s,-9.478901e-07,9.478901e-07
9,TSLA,target_500ms,execution_imbalance,8.177855e-07,8.177855e-07
10,TSLA,target_500ms,return_500ms,-7.484350e-07,7.484350e-07
0,TSLA,target_500ms,spread,6.574364e-07,6.574364e-07


In [20]:
for asset in assets:
    print("\n" + "=" * 70)
    print(asset)
    print("=" * 70)

    for target in target_cols[asset]:
        print(f"\n{target}")

        display(
            ridge_coef_df[
                (ridge_coef_df["asset"] == asset) &
                (ridge_coef_df["target"] == target)
            ]
            .sort_values("abs_coefficient", ascending=False)
            [["feature", "coefficient", "abs_coefficient"]]
            .head(8)
        )


TSLA

target_500ms


,feature,coefficient,abs_coefficient
5,microprice_deviation,3.530797e-06,3.530797e-06
8,normalized_book_flow,1.256795e-06,1.256795e-06
12,return_2s,9.750291e-07,9.750291e-07
14,return_10s,-9.478901e-07,9.478901e-07
9,execution_imbalance,8.177855e-07,8.177855e-07
10,return_500ms,-7.484350e-07,7.484350e-07
0,spread,6.574364e-07,6.574364e-07
3,bid_depth_5,-6.559863e-07,6.559863e-07



target_2s


,feature,coefficient,abs_coefficient
24,microprice_deviation,0.000009,0.000009
20,imbalance_l1,-0.000004,0.000004
33,return_10s,-0.000003,0.000003
36,avg_events_5s,0.000003,0.000003
22,bid_depth_5,-0.000002,0.000002
19,spread,0.000002,0.000002
21,imbalance_l5,0.000001,0.000001
37,avg_executions_5s,-0.000001,0.000001



target_10s


...

,feature,coefficient,abs_coefficient
123,execution_imbalance,0.000007,0.000007
124,return_500ms,-0.000007,0.000007
116,imbalance_l5,0.000006,0.000006
131,avg_events_5s,-0.000006,0.000006
127,return_5s,-0.000005,0.000005
126,return_2s,-0.000004,0.000004
132,avg_executions_5s,0.000004,0.000004
117,bid_depth_5,-0.000003,0.000003



target_10s


,feature,coefficient,abs_coefficient
135,imbalance_l5,0.000011,0.000011
142,execution_imbalance,0.000009,0.000009
148,volatility_5s,-0.000008,0.000008
143,return_500ms,-0.000007,0.000007
151,avg_executions_5s,0.000007,0.000007
136,bid_depth_5,-0.000007,0.000007
150,avg_events_5s,-0.000007,0.000007
137,ask_depth_5,0.000005,0.000005



target_30s


,feature,coefficient,abs_coefficient
154,imbalance_l5,0.000013,0.000013
155,bid_depth_5,-0.000011,0.000011
161,execution_imbalance,0.000010,0.000010
153,imbalance_l1,0.000009,0.000009
162,return_500ms,-0.000008,0.000008
152,spread,0.000006,0.000006
166,return_10s,-0.000005,0.000005
169,avg_events_5s,-0.000004,0.000004


### Feature-analysis observations

The standardized Ridge coefficients reveal meaningful differences across assets
and prediction horizons.

For TSLA, microprice deviation is particularly important at the shortest
horizons, while recent market activity becomes more prominent at the longer
10-second horizon. For AMZN, order-book imbalance is among the strongest
short-horizon predictors, whereas activity, recent returns, volatility, and
deeper-book imbalance become more relevant at 10 seconds. For BRK.B, execution
imbalance and deeper-book imbalance play relatively important roles.

Overall, the results suggest that predictive information is not concentrated in
a single universal feature. The relative importance of order-book state,
order-flow, recent returns, and activity measures varies across both assets and
forecast horizons.

Because several predictors are correlated, coefficient magnitudes are interpreted
as descriptions of the fitted linear models rather than as causal effects.

## 6. Final Out-of-Sample Evaluation

The final evaluation is performed on March 20, which has not been used for
feature selection, model fitting, or hyperparameter selection.

For each asset and forecast horizon, the Ridge regularization parameter selected
using the March 19 validation set is fixed. The model is then refitted using the
combined March 16–19 data and evaluated once on March 20.

The final results are compared with simple benchmark predictions to determine
whether the learned model provides genuine out-of-sample predictive value.

In [21]:
final_train = {}

for asset in assets:
    final_train[asset] = pd.concat([
        splits[asset]["train"],
        splits[asset]["val"]
    ]).sort_index()

    print(
        asset,
        "Final train:", final_train[asset].shape,
        "| Test:", splits[asset]["test"].shape
    )

TSLA Final train: (187200, 75) | Test: (46800, 75)
AMZN Final train: (187200, 75) | Test: (46800, 75)
BRK.B Final train: (187200, 75) | Test: (46800, 75)


In [22]:
final_results = []

for asset in assets:

    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            final_train[asset],
            model_features,
            target
        )

        X_test, y_test = make_xy(
            splits[asset]["test"],
            model_features,
            target
        )

        alpha = best_ridge.loc[
            (best_ridge["asset"] == asset) &
            (best_ridge["target"] == target),
            "alpha"
        ].iloc[0]

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model = Ridge(alpha=alpha)
        model.fit(X_train_scaled, y_train)

        pred = model.predict(X_test_scaled)

        final_results.append({
            "asset": asset,
            "target": target,
            "model": "Ridge",
            "alpha": alpha,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
            "R2": r2_score(y_test, pred),
            "Directional_Accuracy": np.mean(
                np.sign(pred) == np.sign(y_test)
            )
        })

final_results_df = pd.DataFrame(final_results)

display(final_results_df)

,asset,target,model,alpha,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Ridge,0.01,0.000055,0.000085,0.001522,0.445880
1,TSLA,target_2s,Ridge,100.00,0.000117,0.000167,-0.001751,0.490692
2,TSLA,target_10s,Ridge,100.00,0.000272,0.000378,-0.010273,0.504067
3,AMZN,target_500ms,Ridge,0.01,0.000044,0.000071,0.011281,0.385973
4,AMZN,target_2s,Ridge,100.00,0.000095,0.000139,0.001451,0.474174
5,AMZN,target_10s,Ridge,100.00,0.000219,0.000308,-0.007414,0.489897
6,BRK.B,target_5s,Ridge,100.00,0.000082,0.000155,0.008631,0.305521
7,BRK.B,target_10s,Ridge,100.00,0.000129,0.000214,0.002165,0.402611
8,BRK.B,target_30s,Ridge,100.00,0.000254,0.000364,0.002174,0.488046


In [23]:
test_baseline_results = []

for asset in assets:

    for target in target_cols[asset]:

        _, y_train = make_xy(
            final_train[asset],
            model_features,
            target
        )

        _, y_test = make_xy(
            splits[asset]["test"],
            model_features,
            target
        )

        train_mean = y_train.mean()
        pred = np.full(len(y_test), train_mean)

        test_baseline_results.append({
            "asset": asset,
            "target": target,
            "model": "Train Mean",
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
            "R2": r2_score(y_test, pred),
            "Directional_Accuracy": np.mean(
                np.sign(pred) == np.sign(y_test)
            )
        })

test_baseline_df = pd.DataFrame(test_baseline_results)

display(test_baseline_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Train Mean,0.000054,0.000085,-3.071929e-05,0.423843
1,TSLA,target_2s,Train Mean,0.000117,0.000167,-1.282233e-04,0.481321
2,TSLA,target_10s,Train Mean,0.000271,0.000377,-6.960081e-04,0.499294
3,AMZN,target_500ms,Train Mean,0.000043,0.000071,-2.463588e-06,0.337277
4,AMZN,target_2s,Train Mean,0.000095,0.000139,-1.200533e-05,0.440795
5,AMZN,target_10s,Train Mean,0.000218,0.000307,-8.721545e-05,0.475985
6,BRK.B,target_5s,Train Mean,0.000079,0.000156,-8.538988e-07,0.304537
7,BRK.B,target_10s,Train Mean,0.000128,0.000215,-2.483251e-06,0.407727
8,BRK.B,target_30s,Train Mean,0.000255,0.000365,-2.920305e-05,0.471594


In [24]:
final_comparison = pd.concat(
    [
        test_baseline_df,
        final_results_df
    ],
    ignore_index=True
)

final_comparison = final_comparison.sort_values(
    ["asset", "target", "model"]
)

display(final_comparison)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy,alpha
14,AMZN,target_10s,Ridge,0.000219,0.000308,-7.414061e-03,0.489897,100.00
5,AMZN,target_10s,Train Mean,0.000218,0.000307,-8.721545e-05,0.475985,NaN
13,AMZN,target_2s,Ridge,0.000095,0.000139,1.451099e-03,0.474174,100.00
4,AMZN,target_2s,Train Mean,0.000095,0.000139,-1.200533e-05,0.440795,NaN
12,AMZN,target_500ms,Ridge,0.000044,0.000071,1.128117e-02,0.385973,0.01
3,AMZN,target_500ms,Train Mean,0.000043,0.000071,-2.463588e-06,0.337277,NaN
16,BRK.B,target_10s,Ridge,0.000129,0.000214,2.165460e-03,0.402611,100.00
7,BRK.B,target_10s,Train Mean,0.000128,0.000215,-2.483251e-06,0.407727,NaN
17,BRK.B,target_30s,Ridge,0.000254,0.000364,2.173561e-03,0.488046,100.00
8,BRK.B,target_30s,Train Mean,0.000255,0.000365,-2.920305e-05,0.471594,NaN


In [25]:
comparison_summary = []

for asset in assets:
    for target in target_cols[asset]:

        ridge = final_results_df[
            (final_results_df["asset"] == asset) &
            (final_results_df["target"] == target)
        ].iloc[0]

        baseline = test_baseline_df[
            (test_baseline_df["asset"] == asset) &
            (test_baseline_df["target"] == target)
        ].iloc[0]

        comparison_summary.append({
            "asset": asset,
            "target": target,

            "ridge_R2": ridge["R2"],

            "RMSE_improvement_pct":
                100 * (baseline["RMSE"] - ridge["RMSE"])
                / baseline["RMSE"],

            "MAE_improvement_pct":
                100 * (baseline["MAE"] - ridge["MAE"])
                / baseline["MAE"],

            "ridge_directional_accuracy":
                ridge["Directional_Accuracy"],

            "baseline_directional_accuracy":
                baseline["Directional_Accuracy"]
        })

comparison_summary_df = pd.DataFrame(comparison_summary)

display(
    comparison_summary_df.round({
        "ridge_R2": 6,
        "RMSE_improvement_pct": 4,
        "MAE_improvement_pct": 4,
        "ridge_directional_accuracy": 4,
        "baseline_directional_accuracy": 4
    })
)

,asset,target,ridge_R2,RMSE_improvement_pct,MAE_improvement_pct,ridge_directional_accuracy,baseline_directional_accuracy
0,TSLA,target_500ms,0.001522,0.0776,-0.4858,0.4459,0.4238
1,TSLA,target_2s,-0.001751,-0.0811,-0.0885,0.4907,0.4813
2,TSLA,target_10s,-0.010273,-0.4774,-0.1028,0.5041,0.4993
3,AMZN,target_500ms,0.011281,0.5658,-2.2351,0.3860,0.3373
4,AMZN,target_2s,0.001451,0.0732,-0.1662,0.4742,0.4408
5,AMZN,target_10s,-0.007414,-0.3656,-0.2074,0.4899,0.4760
6,BRK.B,target_5s,0.008631,0.4326,-3.4104,0.3055,0.3045
7,BRK.B,target_10s,0.002165,0.1085,-1.4868,0.4026,0.4077
8,BRK.B,target_30s,0.002174,0.1102,0.1091,0.4880,0.4716


### Final test-set observations

The final out-of-sample evaluation shows that the predictive value of the
microstructure features is modest and strongly dependent on both the asset
and the forecast horizon.

Ridge achieves positive test-set R² for TSLA at 500 ms, AMZN at 500 ms and
2 s, and BRK.B at 5 s, 10 s, and 30 s. The strongest results are obtained
for AMZN at 500 ms (R² ≈ 0.0113) and BRK.B at 5 s (R² ≈ 0.0086).

However, the improvements over the training-mean baseline are small. The persistence baseline, corresponding to a zero future-return forecast, also provides essentially no explanatory power, reinforcing that any
improvement achieved by the learned models is modest.
The largest RMSE reductions are approximately 0.57% for AMZN at 500 ms
and 0.43% for BRK.B at 5 s. Several other positive-R² cases improve RMSE
by only around 0.1% or less. Ridge performs worse than the baseline for
TSLA at 2 s and 10 s and for AMZN at 10 s.

The MAE comparison is even more conservative: only BRK.B at 30 s shows
an improvement in MAE, while the other Ridge models have slightly or
substantially higher MAE than the baseline.

Directional accuracy also varies substantially across assets and horizons.
Because high-frequency returns contain many zero observations, directional
accuracy should not be interpreted as a simple comparison against a
universal 50% benchmark.

Overall, the experiment provides evidence of weak out-of-sample predictive
structure in selected asset-horizon combinations rather than a strong or
universal forecasting relationship. The results also demonstrate the
importance of evaluating high-frequency prediction models against simple
baselines and on a strictly held-out trading day.

March 20 was used only for final evaluation and was not used for feature
selection, hyperparameter selection, or model selection.

## 7. Classification Experiment

In addition to predicting future returns as a continuous variable, we formulate
a three-class directional prediction problem:

- **DOWN**: sufficiently negative future return
- **STATIONARY**: small price movement
- **UP**: sufficiently positive future return

Classification is evaluated at one representative horizon for each asset:

- TSLA: 500 ms
- AMZN: 500 ms
- BRK.B: 5 s

These horizons correspond to the shortest selected prediction horizons for
each asset and allow the same future price movement to be studied from both
regression and classification perspectives.

Classification thresholds are determined using training data only. The held-out
test day (March 20) is not used to select the thresholds, features, models, or
hyperparameters.

In [26]:
classification_targets = {
    "TSLA": "target_500ms",
    "AMZN": "target_500ms",
    "BRK.B": "target_5s",
}

classification_targets

{'TSLA': 'target_500ms', 'AMZN': 'target_500ms', 'BRK.B': 'target_5s'}

In [27]:
classification_distribution_rows = []

for asset, target in classification_targets.items():

    train_df = splits[asset]["train"]

    y = train_df[target].dropna()

    classification_distribution_rows.append({
        "asset": asset,
        "target": target,
        "count": len(y),
        "mean": y.mean(),
        "std": y.std(),
        "median": y.median(),
        "mean_abs_return": y.abs().mean(),
        "zero_pct": (y == 0).mean() * 100,
        "q10": y.quantile(0.10),
        "q25": y.quantile(0.25),
        "q33": y.quantile(1/3),
        "q67": y.quantile(2/3),
        "q75": y.quantile(0.75),
        "q90": y.quantile(0.90),
    })

classification_distribution_df = pd.DataFrame(
    classification_distribution_rows
)

display(classification_distribution_df)

,asset,target,count,mean,std,median,mean_abs_return,zero_pct,q10,q25,q33,q67,q75,q90
0,TSLA,target_500ms,140397,-5.602759e-08,0.000072,0.0,0.000042,23.467026,-0.000063,-0.000025,-0.000013,0.000013,0.000025,0.000063
1,AMZN,target_500ms,140397,5.744647e-08,0.000068,0.0,0.000038,38.396832,-0.000070,-0.000024,0.000000,0.000000,0.000024,0.000070
2,BRK.B,target_5s,140370,-1.243502e-06,0.000134,0.0,0.000066,41.823039,-0.000122,-0.000020,0.000000,0.000000,0.000010,0.000113


### 7.1 Classification threshold selection

A simple sign-based classification would assign every nonzero return to either
UP or DOWN. This is undesirable because high-frequency prices exhibit many
zero returns and very small discrete price changes.

The training data show particularly large zero-return fractions for AMZN and
BRK.B. Therefore, classification thresholds are based on the empirical
nonzero return magnitudes observed in the training period rather than on
validation or test data.

In [28]:
movement_rows = []

for asset, target in classification_targets.items():

    y = splits[asset]["train"][target].dropna()

    abs_nonzero = y[y != 0].abs()

    movement_rows.append({
        "asset": asset,
        "target": target,
        "nonzero_count": len(abs_nonzero),
        "min_nonzero": abs_nonzero.min(),
        "q01_nonzero": abs_nonzero.quantile(0.01),
        "q05_nonzero": abs_nonzero.quantile(0.05),
        "q10_nonzero": abs_nonzero.quantile(0.10),
        "median_nonzero": abs_nonzero.median(),
    })

movement_df = pd.DataFrame(movement_rows)

display(movement_df)

,asset,target,nonzero_count,min_nonzero,q01_nonzero,q05_nonzero,q10_nonzero,median_nonzero
0,TSLA,target_500ms,107450,2.220446e-16,0.000012,0.000013,0.000013,0.000038
1,AMZN,target_500ms,86489,1.110223e-16,0.000023,0.000023,0.000023,0.000047
2,BRK.B,target_5s,81663,1.110223e-16,0.000010,0.000010,0.000010,0.000081


In [29]:
threshold_balance_rows = []

for asset, target in classification_targets.items():

    y = splits[asset]["train"][target].dropna()
    abs_nonzero = y[y != 0].abs()

    candidate_thresholds = {
        "zero": 0.0,
        "min_nonzero": abs_nonzero.min(),
        "q05_nonzero": abs_nonzero.quantile(0.05),
        "q10_nonzero": abs_nonzero.quantile(0.10),
    }

    for threshold_name, tau in candidate_thresholds.items():

        down_pct = (y < -tau).mean() * 100
        stationary_pct = ((y >= -tau) & (y <= tau)).mean() * 100
        up_pct = (y > tau).mean() * 100

        threshold_balance_rows.append({
            "asset": asset,
            "target": target,
            "threshold_type": threshold_name,
            "tau": tau,
            "DOWN_pct": down_pct,
            "STATIONARY_pct": stationary_pct,
            "UP_pct": up_pct,
        })

threshold_balance_df = pd.DataFrame(threshold_balance_rows)

display(threshold_balance_df)

,asset,target,threshold_type,tau,DOWN_pct,STATIONARY_pct,UP_pct
0,TSLA,target_500ms,zero,0.000000e+00,37.723741,23.467026,38.809234
1,TSLA,target_500ms,min_nonzero,2.220446e-16,37.723741,23.585262,38.690998
2,TSLA,target_500ms,q05_nonzero,1.253251e-05,35.915297,27.299729,36.784974
3,TSLA,target_500ms,q10_nonzero,1.255958e-05,34.140331,31.136705,34.722964
4,AMZN,target_500ms,zero,0.000000e+00,30.877440,38.396832,30.725728
5,AMZN,target_500ms,min_nonzero,1.110223e-16,30.844676,38.429596,30.725728
6,AMZN,target_500ms,q05_nonzero,2.336967e-05,29.374559,41.495901,29.129540
7,AMZN,target_500ms,q10_nonzero,2.340030e-05,27.868117,44.562206,27.569677
8,BRK.B,target_5s,zero,0.000000e+00,31.205386,41.823039,26.971575
9,BRK.B,target_5s,min_nonzero,1.110223e-16,31.164779,41.863646,26.971575


The classification threshold for each asset is defined as the 10th percentile
of the absolute nonzero future returns in the training set.

This choice removes very small price changes from the directional classes
while preserving sufficiently large DOWN and UP classes for modeling. The
resulting training distributions remain reasonably balanced without forcing
equal class frequencies.

The thresholds are estimated exclusively from the March 16–18 training period
and are subsequently frozen for validation and test evaluation.

In [30]:
classification_thresholds = {}

for asset, target in classification_targets.items():

    y_train = splits[asset]["train"][target].dropna()
    abs_nonzero = y_train[y_train != 0].abs()

    classification_thresholds[asset] = abs_nonzero.quantile(0.10)

classification_thresholds

{'TSLA': np.float64(1.2559579505356809e-05),
 'AMZN': np.float64(2.340029718496282e-05),
 'BRK.B': np.float64(1.0220925300501525e-05)}

In [31]:
def make_classification_target(returns, threshold):
    """
    Convert future returns into three classes:
        -1 = DOWN
         0 = STATIONARY
         1 = UP

    Missing future returns remain NaN.
    """
    labels = pd.Series(
        np.nan,
        index=returns.index,
        dtype=float
    )

    valid = returns.notna()

    labels.loc[valid & (returns < -threshold)] = -1
    labels.loc[valid & (returns >= -threshold) & (returns <= threshold)] = 0
    labels.loc[valid & (returns > threshold)] = 1

    return labels

In [32]:
for asset, target in classification_targets.items():

    threshold = classification_thresholds[asset]

    for split_name in ["train", "val", "test"]:

        df = splits[asset][split_name]

        df["classification_target"] = make_classification_target(
            df[target],
            threshold
        )

In [33]:
class_distribution_rows = []

label_names = {
    -1: "DOWN",
     0: "STATIONARY",
     1: "UP"
}

for asset in classification_targets:

    for split_name in ["train", "val", "test"]:

        df = splits[asset][split_name]

        y = df["classification_target"].dropna()

        counts = y.value_counts()
        percentages = y.value_counts(normalize=True) * 100

        for label, class_name in label_names.items():

            class_distribution_rows.append({
                "asset": asset,
                "split": split_name,
                "class": class_name,
                "count": counts.get(label, 0),
                "percentage": percentages.get(label, 0.0)
            })

class_distribution_df = pd.DataFrame(class_distribution_rows)

display(class_distribution_df)

,asset,split,class,count,percentage
0,TSLA,train,DOWN,47932,34.140331
1,TSLA,train,STATIONARY,43715,31.136705
2,TSLA,train,UP,48750,34.722964
3,TSLA,val,DOWN,19985,42.703904
4,TSLA,val,STATIONARY,7377,15.763157
5,TSLA,val,UP,19437,41.532939
6,TSLA,test,DOWN,19796,42.300049
7,TSLA,test,STATIONARY,7689,16.429838
8,TSLA,test,UP,19314,41.270113
9,AMZN,train,DOWN,39126,27.868117


In [34]:
class_distribution_pivot = (
    class_distribution_df
    .pivot_table(
        index=["asset", "split"],
        columns="class",
        values="percentage"
    )
    .reset_index()
)

display(class_distribution_pivot)

class,asset,split,DOWN,STATIONARY,UP
0,AMZN,test,34.237911,32.171628,33.590461
1,AMZN,train,27.868117,44.562206,27.569677
2,AMZN,val,34.823394,30.970747,34.205859
3,BRK.B,test,30.378286,41.938448,27.683266
4,BRK.B,train,27.863504,47.643371,24.493125
5,BRK.B,val,34.357769,34.695448,30.946784
6,TSLA,test,42.300049,16.429838,41.270113
7,TSLA,train,34.140331,31.136705,34.722964
8,TSLA,val,42.703904,15.763157,41.532939


### Classification target definition

Three return classes are defined using asset-specific thresholds estimated
exclusively from the training period. For each asset, the threshold is the
10th percentile of the absolute nonzero future returns in the training set.

Returns below `-tau` are labeled DOWN, returns between `-tau` and `tau`
are labeled STATIONARY, and returns above `tau` are labeled UP.

The same frozen threshold is applied to the training, validation, and test
sets. The resulting class distributions show noticeable day-to-day variation,
particularly for TSLA, whose STATIONARY share falls from approximately 31%
in training to 16% on the validation and test days. This distribution shift
is retained rather than corrected using future information.

In [36]:
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add a model column consistently if needed
regression_validation_results = pd.concat(
    [
        baseline_results_df,
        mean_baseline_results_df,
        ridge_results_df,
        hgb_results_df,
    ],
    ignore_index=True,
    sort=False
)

display(regression_validation_results)

regression_validation_results.to_csv(
    OUTPUT_DIR / "regression_validation_results.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "regression_validation_results.csv"
)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy,alpha
0,TSLA,target_500ms,Persistence,0.000055,0.000086,-0.000003,0.156080,NaN
1,TSLA,target_2s,Persistence,0.000120,0.000175,-0.000012,0.050026,NaN
2,TSLA,target_10s,Persistence,0.000273,0.000381,-0.000056,0.018236,NaN
3,AMZN,target_500ms,Persistence,0.000045,0.000073,-0.000012,0.307773,NaN
4,AMZN,target_2s,Persistence,0.000098,0.000143,-0.000050,0.115436,NaN
...,...,...,...,...,...,...,...,...
67,AMZN,target_2s,HistGradientBoosting,0.000098,0.000143,0.000569,0.468183,NaN
68,AMZN,target_10s,HistGradientBoosting,0.000225,0.000316,0.003179,0.496554,NaN
69,BRK.B,target_5s,HistGradientBoosting,0.000078,0.000140,-0.004325,0.348235,NaN
70,BRK.B,target_10s,HistGradientBoosting,0.000123,0.000192,0.003591,0.437286,NaN


Saved: ../outputs/regression_validation_results.csv


In [38]:
display(final_results_df)

,asset,target,model,alpha,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Ridge,0.01,0.000055,0.000085,0.001522,0.445880
1,TSLA,target_2s,Ridge,100.00,0.000117,0.000167,-0.001751,0.490692
2,TSLA,target_10s,Ridge,100.00,0.000272,0.000378,-0.010273,0.504067
3,AMZN,target_500ms,Ridge,0.01,0.000044,0.000071,0.011281,0.385973
4,AMZN,target_2s,Ridge,100.00,0.000095,0.000139,0.001451,0.474174
5,AMZN,target_10s,Ridge,100.00,0.000219,0.000308,-0.007414,0.489897
6,BRK.B,target_5s,Ridge,100.00,0.000082,0.000155,0.008631,0.305521
7,BRK.B,target_10s,Ridge,100.00,0.000129,0.000214,0.002165,0.402611
8,BRK.B,target_30s,Ridge,100.00,0.000254,0.000364,0.002174,0.488046


In [39]:
display(test_baseline_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Train Mean,0.000054,0.000085,-3.071929e-05,0.423843
1,TSLA,target_2s,Train Mean,0.000117,0.000167,-1.282233e-04,0.481321
2,TSLA,target_10s,Train Mean,0.000271,0.000377,-6.960081e-04,0.499294
3,AMZN,target_500ms,Train Mean,0.000043,0.000071,-2.463588e-06,0.337277
4,AMZN,target_2s,Train Mean,0.000095,0.000139,-1.200533e-05,0.440795
5,AMZN,target_10s,Train Mean,0.000218,0.000307,-8.721545e-05,0.475985
6,BRK.B,target_5s,Train Mean,0.000079,0.000156,-8.538988e-07,0.304537
7,BRK.B,target_10s,Train Mean,0.000128,0.000215,-2.483251e-06,0.407727
8,BRK.B,target_30s,Train Mean,0.000255,0.000365,-2.920305e-05,0.471594


In [40]:
display(final_comparison)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy,alpha
14,AMZN,target_10s,Ridge,0.000219,0.000308,-7.414061e-03,0.489897,100.00
5,AMZN,target_10s,Train Mean,0.000218,0.000307,-8.721545e-05,0.475985,NaN
13,AMZN,target_2s,Ridge,0.000095,0.000139,1.451099e-03,0.474174,100.00
4,AMZN,target_2s,Train Mean,0.000095,0.000139,-1.200533e-05,0.440795,NaN
12,AMZN,target_500ms,Ridge,0.000044,0.000071,1.128117e-02,0.385973,0.01
3,AMZN,target_500ms,Train Mean,0.000043,0.000071,-2.463588e-06,0.337277,NaN
16,BRK.B,target_10s,Ridge,0.000129,0.000214,2.165460e-03,0.402611,100.00
7,BRK.B,target_10s,Train Mean,0.000128,0.000215,-2.483251e-06,0.407727,NaN
17,BRK.B,target_30s,Ridge,0.000254,0.000364,2.173561e-03,0.488046,100.00
8,BRK.B,target_30s,Train Mean,0.000255,0.000365,-2.920305e-05,0.471594,NaN


In [41]:
from pathlib import Path

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

final_comparison.to_csv(
    OUTPUT_DIR / "regression_test_results.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "regression_test_results.csv")

Saved: ../outputs/regression_test_results.csv


In [42]:
test_persistence_results = []

for asset in assets:

    for target in target_cols[asset]:

        _, y_test = make_xy(
            splits[asset]["test"],
            model_features,
            target
        )

        # Persistence in price space:
        # future price = current price
        # therefore future log return = 0
        pred = np.zeros(len(y_test))

        test_persistence_results.append({
            "asset": asset,
            "target": target,
            "model": "Persistence",
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, pred)
            ),
            "R2": r2_score(y_test, pred),
            "Directional_Accuracy": np.mean(
                np.sign(pred) == np.sign(y_test)
            )
        })

test_persistence_df = pd.DataFrame(
    test_persistence_results
)

display(test_persistence_df)

,asset,target,model,MAE,RMSE,R2,Directional_Accuracy
0,TSLA,target_500ms,Persistence,0.000054,0.000085,-4.205185e-05,0.162562
1,TSLA,target_2s,Persistence,0.000117,0.000167,-1.734538e-04,0.049234
2,TSLA,target_10s,Persistence,0.000271,0.000377,-9.008621e-04,0.017723
3,AMZN,target_500ms,Persistence,0.000043,0.000071,-4.233716e-07,0.319369
4,AMZN,target_2s,Persistence,0.000095,0.000139,-2.062356e-06,0.114109
5,AMZN,target_10s,Persistence,0.000218,0.000306,-2.103226e-05,0.041588
6,BRK.B,target_5s,Persistence,0.000079,0.000156,-4.089163e-05,0.417911
7,BRK.B,target_10s,Persistence,0.000127,0.000215,-8.344489e-05,0.220783
8,BRK.B,target_30s,Persistence,0.000254,0.000365,-2.222634e-04,0.036889


In [43]:
# Add Persistence results to the existing final test results
updated_regression_test = pd.concat(
    [
        final_results_df,
        test_baseline_df,
        test_persistence_df
    ],
    ignore_index=True
)

updated_regression_test.to_csv(
    "../outputs/regression_test_results.csv",
    index=False
)

print(updated_regression_test["model"].value_counts())

model
Ridge          9
Train Mean     9
Persistence    9
Name: count, dtype: int64
